💡 **Environment:** `clamp-analyses`

# CRISPR-Cas9 stratagenic network and TWAS recovery

This notebook contains and executes the plotting code. A *missed gene* is a CRISPR-supported gene that is strongly represented in a trait-associated latent program (LV), but lacks a significant single-gene TWAS association for that same trait. Each coloured ellipse is an LV community; its lower label gives the selected pathway and cell-type context. Stars mark CRISPR-supported recovered genes and circles mark other high-loading LV genes. Solid coloured lines to a trait node require nominal same-trait S-MultiXcan evidence (*p* < 0.05). Thin grey dashed lines within an LV indicate gene pairs co-occurring in a canonical pathway significantly enriched for that LV (ORA adjusted *p* < 0.05). Thicker dashed lines in the trait colour connect the same displayed gene across LV communities for that trait. The TWAS bubbles encode gene-level S-MultiXcan −log₁₀(*p*), capped at 60; bold rows are the CRISPR-supported recovered genes.


In [ ]:
# Render CRISPR-Cas9 stratagenic network, TWAS, and combined figures.

import textwrap
from pathlib import Path
from IPython.display import HTML, display

import matplotlib
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Ellipse
from scipy.stats import norm

matplotlib.use("Agg")
import matplotlib.pyplot as plt


OUT = Path(snakemake.input["mapping"]).parent
Z = pd.read_csv(snakemake.input["z"], index_col=0)
TW = pd.read_pickle(snakemake.input["smultixcan"])
GLS = pd.read_csv(snakemake.input["gls"], sep="\t")
ORA = pd.read_csv(snakemake.input["canonical_ora"])
ORA = ORA[ORA["p.adjust"] < 0.05]
SYMBOL_MAP = pd.read_csv(snakemake.input["mapping"]).dropna().drop_duplicates("SYMBOL").set_index("SYMBOL")["ENSEMBL"]

GROUPS = [
    ("Alzheimer’s", "#8870AA", [("LV1226", "DGAT2", "AD-"), ("LV389", "PCYT2", "20110_10-"), ("LV627", "DGAT2", "20110_10-")], "Alzheimer’s"),
    ("Diabetes", "#477BA8", [("LV1167", "SOX9", "20002_1220-"), ("LV660", "SOX9", "20002_1222-"), ("LV902", "PTEN", "20107_9-")], "Diabetes"),
    ("High cholesterol", "#378A80", [("LV1226", "DGAT2", "20002_1473-"), ("LV1227", "SOX9", "20002_1473-")], "High cholesterol"),
    ("Coronary diseases", "#C18539", [("LV624", "DGAT2", "I25-"), ("LV294", "PTEN", "20002_1074-")], "Coronary diseases"),
    ("Fat mass", "#C8668C", [("LV1421", "HILPDA", "23120_raw-")], "Fat mass"),
    ("Biliary diseases", "#869746", [("LV588", "DGAT2", "K80-")], "Biliary diseases"),
]

LABELS = {
    "LV1226": "Lipoprotein transport / hepatocyte",
    "LV1227": "Cholesterol homeostasis / hepatocyte",
    "LV389": "Phospholipid synthesis / hepatic cell lines",
    "LV588": "Triglyceride synthesis / mixed cell markers",
    "LV627": "Fatty-acid oxidation / hepatocyte",
    "LV1167": "Insulin–FOXA signaling / hepatocyte",
    "LV660": "Antigen presentation / dendritic cell",
    "LV902": "Unresolved pathway / adipocyte",
    "LV624": "Growth-factor signaling / mixed cell markers",
    "LV294": "Innate immune sensing / immune cells",
    "LV1421": "IL12–p38 signaling / mixed cell markers",
}

GENE_IDS = {
    "APOE": "ENSG00000130203", "APOC1": "ENSG00000130208", "APOC2": "ENSG00000234906",
    "LPL": "ENSG00000175445", "LDLR": "ENSG00000130164", "LIPC": "ENSG00000166035",
    "DGAT2": "ENSG00000062282", "SOX9": "ENSG00000125398", "PCYT2": "ENSG00000185813",
    "PTEN": "ENSG00000171862", "HILPDA": "ENSG00000135245",
}


def twas_pvalue(gene, trait_column):
    ensembl = SYMBOL_MAP.get(gene)
    if pd.isna(ensembl) or ensembl not in TW.index:
        return None
    return float(2 * norm.sf(abs(TW.loc[ensembl, trait_column])))


def render_network(show=False):
    edge_rows, gene_rows, trait_rows, shared_rows, summary_rows = [], [], [], [], []
    fig, axes = plt.subplots(2, 3, figsize=(20, 10.7))
    fig.subplots_adjust(left=.02, right=.99, top=.985, bottom=.025, hspace=.055, wspace=.045)
    for ax, (title, color, modules, phenotype) in zip(axes.flat, GROUPS):
        ax.axis("off")
        ax.set_xlim(0, 11.5)
        positions = {1: [(3.6, 8.5)], 2: [(3.2, 12.0), (4.7, 4.4)], 3: [(4.7, 14.0), (2.6, 8.1), (4.7, 2.2)]}[len(modules)]
        ax.set_ylim(*{1: (4.2, 12.8), 2: (1.0, 15.4), 3: (.1, 16.5)}[len(modules)])
        anchor = (10.35, 8.25)
        displayed = {}
        for (lv, hit, trait_prefix), (cx, cy) in zip(modules, positions):
            radius = 1.28
            top = list(Z[lv].nlargest(185).index)
            enriched = ORA[ORA.LV == lv].sort_values("p.adjust")
            gene_sets = [set(str(value).split("/")) for value in enriched.geneID]
            trait_column = next(column for column in TW.columns if column.startswith(trait_prefix))
            trait_id = trait_prefix.rstrip("-")
            trait_association = GLS[(GLS.lv == lv) & (GLS.phenotype.astype(str) == trait_id)]
            trait_fdr = trait_association.fdr.iloc[0] if len(trait_association) else np.nan
            trait_description = trait_association.phenotype_desc.iloc[0] if len(trait_association) else ""
            supported = [(gene, pvalue) for gene in top if (pvalue := twas_pvalue(gene, trait_column)) is not None and pvalue < .05]
            genes = [hit] + [gene for gene, _ in supported if gene != hit][:4]
            genes += [gene for gene in top if gene not in genes][:5 - len(genes)]
            ax.add_patch(Ellipse((cx, cy), 3.8, 3.6, facecolor=color, alpha=.13, edgecolor="none"))
            coordinates = {gene: (cx + radius * np.cos(np.pi / 2 + 2 * np.pi * i / len(genes)), cy + radius * np.sin(np.pi / 2 + 2 * np.pi * i / len(genes))) for i, gene in enumerate(genes)}
            for i, gene1 in enumerate(genes):
                for gene2 in genes[i + 1:]:
                    pathways = [str(enriched.iloc[index].ID) for index, gene_set in enumerate(gene_sets) if gene1 in gene_set and gene2 in gene_set]
                    if pathways:
                        ax.plot([coordinates[gene1][0], coordinates[gene2][0]], [coordinates[gene1][1], coordinates[gene2][1]], lw=.8, ls="--", color="#6A7886", alpha=.6, zorder=1)
                        edge_rows.append([title, lv, gene1, gene2, ";".join(pathways)])
            for gene, (x, y) in coordinates.items():
                ax.scatter(x, y, s=180 if gene == hit else 95, marker="*" if gene == hit else "o", color=color, edgecolor="#26333D" if gene == hit else "white", lw=1.1 if gene == hit else .5, zorder=3)
                ax.text(cx + (x - cx) * 1.48, cy + (y - cy) * 1.43, gene, ha="center", va="center", fontsize=12, fontstyle="italic", fontweight="bold" if gene == hit else "normal", zorder=4)
                gene_rows.append([title, lv, gene, int(Z[lv].rank(ascending=False, method="min").loc[gene]), gene == hit])
                displayed.setdefault(gene, []).append((lv, (x, y)))
            for gene, pvalue in supported:
                if gene in coordinates:
                    x, y = coordinates[gene]
                    ax.annotate("", xy=(anchor[0] - .95, anchor[1]), xytext=(x, y), arrowprops=dict(arrowstyle="-", color=color, lw=1.05, alpha=.58), zorder=0)
                    trait_rows.append([title, lv, gene, trait_column, pvalue])
            ax.text(cx, cy + 2.10, lv, ha="center", fontsize=13, fontweight="bold", color=color)
            ax.text(cx, cy - 2.18, "\n".join(textwrap.wrap(LABELS[lv], 39)), ha="center", va="top", fontsize=12, fontweight="bold", color="#33424F")
            summary_rows.append({
                "trait_group": title,
                "LV": lv,
                "CRISPR_gene": hit,
                "trait_id": trait_id,
                "trait_description": trait_description,
                "LV_trait_FDR": trait_fdr,
                "pathway_cell_type": LABELS[lv],
                "displayed_genes": "; ".join(genes),
                "displayed_gene_loading_ranks": "; ".join(f"{gene}:{int(Z[lv].rank(ascending=False, method='min').loc[gene])}" for gene in genes),
                "nominal_same_trait_TWAS_genes": "; ".join(gene for gene, _ in supported),
                "nominal_same_trait_TWAS_pvalues": "; ".join(f"{gene}:{pvalue:.3g}" for gene, pvalue in supported),
            })
        for gene, nodes in displayed.items():
            for index, (lv1, xy1) in enumerate(nodes):
                for lv2, xy2 in nodes[index + 1:]:
                    ax.plot([xy1[0], xy2[0]], [xy1[1], xy2[1]], lw=1.5, ls=(0, (4, 3)), color=color, alpha=.9, zorder=2)
                    shared_rows.append([title, gene, lv1, lv2])
        ax.text(*anchor, phenotype, ha="center", va="center", fontsize=13, fontweight="bold", bbox=dict(boxstyle="round,pad=.6", fc=color, ec="none", alpha=.18))
    pd.DataFrame(edge_rows, columns=["trait_group", "LV", "gene1", "gene2", "shared_enriched_pathways"]).to_csv(OUT / "community_edges.csv", index=False)
    pd.DataFrame(gene_rows, columns=["trait_group", "LV", "gene", "rank", "recovered"]).to_csv(OUT / "community_genes.csv", index=False)
    pd.DataFrame(trait_rows, columns=["trait_group", "LV", "gene", "trait_column", "twas_pvalue"]).to_csv(OUT / "trait_supported_gene_edges.csv", index=False)
    pd.DataFrame(shared_rows, columns=["trait_group", "gene", "LV1", "LV2"]).to_csv(OUT / "inter_lv_shared_gene_edges.csv", index=False)
    summary = pd.DataFrame(summary_rows)
    selected_lvs = summary.LV.drop_duplicates().tolist()
    raw_traits = GLS[(GLS.lv.isin(selected_lvs)) & (GLS.fdr < .05)].copy()
    raw_pathways = ORA[ORA.LV.isin(selected_lvs)].copy()
    raw_traits.to_csv(OUT / "network_lv_significant_traits.csv", index=False)
    raw_pathways.to_csv(OUT / "network_lv_significant_pathways.csv", index=False)
    traits_by_lv = raw_traits.groupby("lv").apply(lambda table: "; ".join(f"{row.phenotype_desc} [{row.phenotype}; FDR={row.fdr:.3g}]" for _, row in table.sort_values("fdr").iterrows()))
    pathways_by_lv = raw_pathways.groupby("LV").apply(lambda table: "; ".join(f"{row.Description} [FDR={row['p.adjust']:.3g}]" for _, row in table.sort_values("p.adjust").iterrows()))
    summary["n_significant_traits"] = summary.LV.map(raw_traits.groupby("lv").size()).fillna(0).astype(int)
    summary["significant_traits"] = summary.LV.map(traits_by_lv).fillna("")
    summary["n_significant_canonical_pathways"] = summary.LV.map(raw_pathways.groupby("LV").size()).fillna(0).astype(int)
    summary["significant_canonical_pathways"] = summary.LV.map(pathways_by_lv).fillna("")
    summary["shared_gene_LV_connections"] = summary.apply(lambda row: "; ".join(f"{edge[1]}:{edge[3] if edge[2] == row.LV else edge[2]}" for edge in shared_rows if edge[0] == row.trait_group and row.LV in edge[2:]), axis=1)
    summary.to_csv(OUT / "network_lv_summary.csv", index=False)
    if show:
        return fig, summary
    plt.close(fig)


def render_twas(show=False):
    genes = list(GENE_IDS)
    traits = [("20002_1473-", "High\ncholesterol"), ("6177_1-", "Cholesterol-\nlowering medication"), ("AD-", "Alzheimer’s\n(self-report)"), ("IGAP_Alzheimer", "Alzheimer’s\n(IGAP-based TWAS)"), ("20110_10-", "Maternal\nAD / dementia"), ("20002_1074-", "Self-reported\nangina"), ("23120_raw-", "Right-arm\nfat mass")]
    rows = []
    for gene, ensembl in GENE_IDS.items():
        for column, (prefix, label) in enumerate(traits):
            trait_column = next(value for value in TW.columns if value.startswith(prefix))
            pvalue = float(2 * norm.sf(abs(TW.loc[ensembl, trait_column])))
            rows.append([gene, label, pvalue, min(-np.log10(max(pvalue, 1e-300)), 60), column])
    table = pd.DataFrame(rows, columns=["gene", "trait", "pvalue", "neg_log10_p_capped60", "column"])
    table.to_csv(OUT / "poster_style_twas_matrix.csv", index=False)
    fig, ax = plt.subplots(figsize=(16, 9.5))
    fig.subplots_adjust(left=.13, right=.89, top=.74, bottom=.16)
    cmap = LinearSegmentedColormap.from_list("poster", ["#DCEFEA", "#1B4D43"])
    for _, row in table.iterrows():
        y = len(genes) - 1 - genes.index(row.gene)
        ax.scatter(row.column, y, s=18 + row.neg_log10_p_capped60 / 60 * 850, c=[row.neg_log10_p_capped60], cmap=cmap, vmin=0, vmax=60, edgecolor="#879B95", linewidth=.8, zorder=3)
    ax.set_xticks(range(len(traits)), [label for _, label in traits], fontsize=11)
    ax.xaxis.tick_top()
    ax.tick_params(length=0, pad=12)
    ax.set_yticks(range(len(genes)), genes[::-1], fontstyle="italic", fontsize=13)
    ax.set_xlim(-.6, len(traits) - .4)
    ax.set_ylim(-.6, len(genes) - .4)
    ax.grid(color="#E4E8E7", lw=.8)
    ax.axhline(4.5, color="#AAB6B2", ls="--", lw=1)
    for label in ax.get_yticklabels():
        if label.get_text() in {"DGAT2", "SOX9", "PCYT2", "PTEN", "HILPDA"}:
            label.set_fontweight("bold")
    colorbar = fig.colorbar(plt.cm.ScalarMappable(norm=plt.Normalize(0, 60), cmap=cmap), ax=ax, fraction=.035, pad=.04)
    colorbar.set_label("S-MultiXcan\n−log₁₀(p), capped at 60", fontsize=11)
    if show:
        return fig
    plt.close(fig)


OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10, "axes.spines.top": False, "axes.spines.right": False, "savefig.facecolor": "white"})
network_figure, lv_information = render_network(show=True)
twas_figure = render_twas(show=True)

def show_table(table):
    html = table.to_html(index=False, escape=True)
    style = """
    <style>
      .full-table-wrap { max-height: 650px; overflow-y: auto; border: 1px solid #ccc; background: #ffffff; }
      table.full-table { border-collapse: collapse; width: 100%; font-size: 13px; background: #ffffff; color: #000000; }
      table.full-table th { position: sticky; top: 0; background: #eef1f5; color: #000000; padding: 6px 8px; border: 1px solid #ccc; text-align: left; }
      table.full-table td { padding: 6px 8px; border: 1px solid #e0e0e0; vertical-align: top; white-space: normal; word-wrap: break-word; background: #ffffff; color: #000000; }
      table.full-table td:first-child, table.full-table th:first-child { white-space: nowrap; font-weight: bold; }
      table.full-table tr:nth-child(even) td { background: #f5f5f5; }
    </style>
    """
    display(HTML(style + '<div class="full-table-wrap">' + html.replace('<table border="1" class="dataframe">', '<table class="full-table">') + '</div>'))


## LV information


In [ ]:
lv_index = lv_information.loc[:, ["trait_group", "LV", "CRISPR_gene"]].drop_duplicates()
show_table(lv_index)


## Significant traits


In [ ]:
significant_traits = pd.read_csv(OUT / "network_lv_significant_traits.csv").rename(columns={"lv": "LV"})
significant_traits = lv_index.merge(significant_traits, on="LV", how="inner")
show_table(significant_traits)


## Significant canonical pathways


In [ ]:
significant_pathways = pd.read_csv(OUT / "network_lv_significant_pathways.csv")
significant_pathways = lv_index.merge(significant_pathways, on="LV", how="inner")
show_table(significant_pathways)


## Displayed LV genes


In [ ]:
displayed_genes = pd.read_csv(OUT / "community_genes.csv")
show_table(displayed_genes)


## Same-trait TWAS-supported genes


In [ ]:
trait_supported_genes = pd.read_csv(OUT / "trait_supported_gene_edges.csv")
show_table(trait_supported_genes)


## Shared genes between LV communities


In [ ]:
shared_gene_connections = pd.read_csv(OUT / "inter_lv_shared_gene_edges.csv")
show_table(shared_gene_connections)


## Stratagenic network


In [ ]:
from io import BytesIO
from IPython.display import HTML, Image, display

network_png = BytesIO()
network_figure.savefig(network_png, format="png", bbox_inches="tight")
display(Image(data=network_png.getvalue()))


## Single-gene TWAS


In [ ]:
twas_png = BytesIO()
twas_figure.savefig(twas_png, format="png", bbox_inches="tight")
display(Image(data=twas_png.getvalue()))


In [ ]:
Path(snakemake.output["complete"]).touch()
